# 03 — Build canonical Silver race state

AIDP operational pipeline notebook. It uses Volumes and governed tables only; the Git checkout is code only.

In [ ]:
from pathlib import Path
import os
import sys
from pyspark.sql import SparkSession, functions as F, Window
spark = SparkSession.builder.getOrCreate()
root = Path.cwd().resolve()
for candidate in (Path(os.environ['AIDP_PROJECT_ROOT']).resolve(), root, *root.parents) if os.getenv('AIDP_PROJECT_ROOT') else (root, *root.parents):
    if (candidate / 'src').is_dir() and (candidate / 'config/settings.yaml').is_file():
        root = candidate
        break
else:
    raise RuntimeError('Project code root not found. Clone the repository; do not use it as a data store.')
sys.path.insert(0, str(root / 'src'))
from aidp_pipeline import LANDING_COUNTS, RAW_SESSIONS, STRUCTURAL_KEYS, landing_paths, load_aidp_config, missing_landing_paths, table_name
config = load_aidp_config(root / 'config/settings.yaml')


In [ ]:
silver = lambda name: table_name(config, 'silver_schema', name)
a = spark.table(silver('analysis_laps'))
pos = spark.table(silver('lap_positions')).select('session','lap_number','driver_id',F.col('position').alias('lap_position'))
gaps = spark.table(silver('lap_gaps')).select('session','lap_number','driver_id','gap_to_leader_seconds')
weather = spark.table(silver('weather'))
keys = ['session','lap_number','driver_id']
if a.count() != LANDING_COUNTS['analysis_laps'] or a.groupBy(*keys).count().filter('count > 1').count():
    raise ValueError('analysis_laps does not have the validated canonical grain')
# Preserve local semantics: weather timestamps are matched to lap hour after the validated -1 hour correction, within 60 seconds.
dates = weather.groupBy('session').agg(F.to_date(F.min('timestamp')).alias('_weather_date'))
base = a.join(pos, keys, 'left').join(gaps, keys, 'left').join(dates, 'session', 'left')
base = base.withColumn('_target', F.expr(f"to_timestamp(concat(date_format(_weather_date, 'yyyy-MM-dd'), ' ', hour)) + INTERVAL {int(config['weather_offset_hours'])} HOURS"))
joined = base.join(weather.select('session',F.col('timestamp').alias('weather_source_timestamp'),'air_temp','humidity','pressure','rain','wind_speed','wind_direction'), 'session', 'left')
joined = joined.withColumn('weather_match_delta_seconds', F.abs(F.unix_timestamp('_target') - F.unix_timestamp('weather_source_timestamp')))
matched = joined.filter(F.col('weather_match_delta_seconds') <= 60)
window = Window.partitionBy(*keys).orderBy(F.col('weather_match_delta_seconds').asc_nulls_last())
matched = matched.withColumn('_weather_rank', F.row_number().over(window)).filter('_weather_rank = 1').drop('_weather_rank')
race = base.drop('_weather_date').join(matched.select(*keys,'weather_source_timestamp','weather_match_delta_seconds','air_temp','humidity','pressure','rain','wind_speed','wind_direction'), keys, 'left')
race = race.select('session','lap_number','driver_id','lap_time_seconds','sector1_seconds','sector2_seconds','sector3_seconds','is_pit_lap','lap_position','gap_to_leader_seconds','weather_source_timestamp','weather_match_delta_seconds','air_temp','humidity','pressure','rain','wind_speed','wind_direction')


In [ ]:
if race.count() != 781 or race.groupBy('session','lap_number','driver_id').count().filter('count > 1').count():
    raise ValueError('race_state canonical validation failed')
if race.filter((F.col('lap_number') <= 0) | (F.col('lap_time_seconds') <= 0) | F.col('driver_id').cast('string').rlike('VIRTUAL_MAX|SYN_') | (F.col('lap_time_seconds') == -999)).count():
    raise ValueError('race_state contains invalid or synthetic historical values')
race.write.mode('overwrite').saveAsTable(table_name(config, 'silver_schema', 'race_state'))
print('Published Silver race_state:', race.count(), 'rows')
